<a href="https://colab.research.google.com/github/WamsyJ/Scaler/blob/main/BuildingResearchAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install openai tavily-python --quiet

# Suppress noisy deprecation warnings for a clean demo
import warnings
warnings.filterwarnings("ignore")

# Import everything we'll need throughout the workshop
import os
import json
import requests
from datetime import datetime
from getpass import getpass
from openai import OpenAI
from tavily import TavilyClient

print("✅ All packages installed and imports ready!")


✅ All packages installed and imports ready!


In [ ]:
# ⚠️ Enter your API keys here
# (In production you'd use environment variables or a secrets manager.
#  For a workshop, getpass() is fine — it hides your input from view.)

# When you run this cell, LOOK at the cell output area for the input prompts.
# Type your key, press Enter. The prompt will appear twice — once for each key.

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API key: ")

# Initialize the clients we'll use throughout
client = OpenAI()
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

print("✅ API keys set, clients initialized!")


Enter your OpenAI API key: ··········
Enter your Tavily API key: ··········
✅ API keys set, clients initialized!


In [ ]:
# A simple, single-turn LLM call
response = client.chat.completions.create(
    model="gpt-4o-mini",          # Fast & cheap model, perfect for learning
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ]
)

# Extract the text response
answer = response.choices[0].message.content
print("🤖 LLM says:", answer)

🤖 LLM says: The capital of France is Paris.


In [ ]:
# Now ask something the LLM *cannot* know from training alone
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the top 3 trending tech news stories TODAY?"}
    ]
)

answer = response.choices[0].message.content
print("🤖 LLM says:", answer)
print()
print("⚠️  Notice: The LLM either made something up (hallucination)")
print("    or admitted it doesn't have real-time info.")
print("    This is EXACTLY the problem agents solve — by giving the LLM TOOLS.")


🤖 LLM says: I’m unable to provide real-time updates or current news stories. However, I recommend checking reliable tech news websites such as TechCrunch, The Verge, or Wired for the latest trending stories. Social media platforms like Twitter or news aggregators like Google News can also help you find the most current tech news. If there's a specific topic you're interested in, I can provide background information or historical context!

⚠️  Notice: The LLM either made something up (hallucination)
    or admitted it doesn't have real-time info.
    This is EXACTLY the problem agents solve — by giving the LLM TOOLS.


In [ ]:
# ──────────────────────────────────────────────────────────
# STEP 1: Define the actual Python function that does the work
# ──────────────────────────────────────────────────────────

def search_web(query: str, max_results: int = 3) -> str:
    """Search the web using Tavily and return formatted results."""
    response = tavily_client.search(
        query=query,
        max_results=max_results,
        search_depth="basic"  # use "advanced" for deeper research (costs more)
    )

    # Format results into a readable string
    output = []
    for i, r in enumerate(response.get("results", []), 1):
        output.append(
            f"{i}. {r['title']}\n"
            f"   {r['content'][:300]}\n"
            f"   URL: {r['url']}"
        )
    return "\n\n".join(output) if output else "No results found."


# Let's test it directly
print("🔍 Testing our search function:\n")
print(search_web("latest tech news India 2026"))


🔍 Testing our search function:

1. Top Trends in 2026 That Will Define India's Tech Adoption
   Discover India tech trends 2026—from AI agents to semiconductor shifts and premium electronics shaping digital adoption.
   URL: https://drita.in/blog/india-tech-trends-2026/

2. Indian Tech Market Update 2026: Trends & Analysis - LinkedIn
   According to Nokia's MBiT 2026 report, India could reach 1 billion 5G subscribers by 2031, driven by rapid adoption and evolving use cases.
   URL: https://www.linkedin.com/posts/budgettechindia_latest-indian-tech-market-update-2026-news-activity-7445427363128983552-_SmL

3. India tech spending growth to dip slightly in 2026: Forrester report
   India's tech spending growth is set to dip slightly to 13.4% in 2026, though it will remain among the highest in Asia, says Forrester.
   URL: https://www.business-standard.com/technology/tech-news/indias-tech-spending-to-dip-slightly-in-2026-forrester-126032600952_1.html


In [ ]:
# ──────────────────────────────────────────────────────────
# STEP 2: Create the tool definition (JSON Schema)
# ──────────────────────────────────────────────────────────
# This is like writing an API spec — you're telling the LLM
# "here's a function you can call, here's what it expects"

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the internet for current information. Use this when you need real-time data, recent news, or facts you're unsure about.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Number of results to return (default 3)",
                        "default": 3
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("✅ Tool defined! Here's what the LLM will see:")
print(json.dumps(tools, indent=2))


✅ Tool defined! Here's what the LLM will see:
[
  {
    "type": "function",
    "function": {
      "name": "search_web",
      "description": "Search the internet for current information. Use this when you need real-time data, recent news, or facts you're unsure about.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "The search query to look up"
          },
          "max_results": {
            "type": "integer",
            "description": "Number of results to return (default 3)",
            "default": 3
          }
        },
        "required": [
          "query"
        ]
      }
    }
  }
]


In [ ]:
# ──────────────────────────────────────────────────────────
# STEP 3: Call the LLM with tools available
# ──────────────────────────────────────────────────────────

messages = [
    {
        "role": "system",
        "content": "You are a helpful research assistant. Use the search_web tool to find current information when needed."
    },
    {
        "role": "user",
        #"content": "What is the capital of France?"
        #"content": "West Bengal Elections 2026?"
        "content": "What are the top 3 trending tech news stories today?"
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,            # ← THIS IS THE KEY ADDITION
    tool_choice="auto"      # Let the LLM decide whether to use tools
)

# Let's inspect what the LLM returned
assistant_message = response.choices[0].message

print("🔍 Did the LLM want to call a tool?", "YES ✅" if assistant_message.tool_calls else "NO")
print()

if assistant_message.tool_calls:
    for tc in assistant_message.tool_calls:
        print(f"📞 Tool Call:")
        print(f"   Function: {tc.function.name}")
        print(f"   Arguments: {tc.function.arguments}")
else:
    print("Response:", assistant_message.content)


🔍 Did the LLM want to call a tool? YES ✅

📞 Tool Call:
   Function: search_web
   Arguments: {"query":"top trending tech news stories","max_results":3}


In [ ]:
# ──────────────────────────────────────────────────────────
# STEP 4: Execute the function and send results back
# ──────────────────────────────────────────────────────────

# Map of function name → actual Python function
available_functions = {
    "search_web": search_web
}

# Process each tool call the LLM requested
if assistant_message.tool_calls:
    # First, add the assistant's message (with tool calls) to conversation
    messages.append(assistant_message)

    for tool_call in assistant_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

        print(f"⚡ Executing: {function_name}({function_args})")
        print("-" * 50)

        # Actually call the function
        function_response = available_functions[function_name](**function_args)

        print(f"📄 Got results ({len(function_response)} chars)")
        print()

        # Send the result back to the LLM as a "tool" message
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": function_response
        })

    # Now call the LLM again — this time with the search results in context
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    print("=" * 50)
    print("🤖 FINAL ANSWER (informed by real search results):")
    print("=" * 50)
    print(final_response.choices[0].message.content)


⚡ Executing: search_web({'query': 'top trending tech news stories', 'max_results': 3})
--------------------------------------------------
📄 Got results (1167 chars)

🤖 FINAL ANSWER (informed by real search results):
Here are three of the top trending tech news stories today:

1. **Amazon's New AI Initiatives** - Amazon is ramping up its artificial intelligence offerings, with new products aimed at enhancing its cloud computing services. The company is focusing on making AI tools more accessible to businesses of all sizes.

2. **Meta's Workforce Changes** - Meta (Facebook's parent company) has announced significant changes to its workforce structure as it integrates artificial intelligence across its platforms. These changes are part of the initiative to enhance user experience and improve content moderation.

3. **Apple's iPhone Updates** - Apple is set to release an update for its iPhone models, which will include new features and security enhancements. This update is highly anticipat

In [ ]:
# ══════════════════════════════════════════════════════════
# TOOL 1: Web Search (Tavily)
# ══════════════════════════════════════════════════════════
# We already defined search_web above. Let's enhance it slightly
# to support more results for the full agent.

def search_web(query: str, max_results: int = 5) -> str:
    """Search the web using Tavily and return formatted results."""
    response = tavily_client.search(
        query=query,
        max_results=max_results,
        search_depth="basic"
    )
    output = []
    for i, r in enumerate(response.get("results", []), 1):
        output.append(
            f"{i}. **{r['title']}**\n"
            f"   {r['content'][:400]}\n"
            f"   URL: {r['url']}"
        )
    return "\n\n".join(output) if output else "No results found."


# ══════════════════════════════════════════════════════════
# TOOL 2: Read a Webpage
# ══════════════════════════════════════════════════════════
# Tavily's extract API is built for AI agents — it gives us
# clean, structured text from any URL (handles JS, paywalls, etc.)

def get_webpage_text(url: str) -> str:
    """Fetch and extract clean text content from a webpage URL."""
    try:
        response = tavily_client.extract(urls=[url])
        results = response.get("results", [])
        if not results:
            return f"Could not extract content from {url}"

        content = results[0].get("raw_content", "")

        # Truncate to ~3000 chars to stay within context limits
        if len(content) > 10000:
            content = content[:3000] + "... [truncated]"

        return content if content else "Page had no extractable content."
    except Exception as e:
        return f"Error fetching page: {str(e)}"


# ══════════════════════════════════════════════════════════
# TOOL 3: Save Final Report
# ══════════════════════════════════════════════════════════

def save_report(title: str, content: str) -> str:
    """Save the research report to a markdown file."""
    filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
    with open(filename, "w") as f:
        f.write(f"# {title}\n\n")
        f.write(f"*Generated on {datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n")
        f.write(content)
    return f"Report saved as '{filename}'"


# Quick sanity test
print("🔍 Testing search_web...")
print(search_web("US Iran War", max_results=2))
print("-" * 50)
print("🔍 Testing get_webpage_text...")
print(get_webpage_text("https://www.scaler.com/event/how-to-crack-ai-roles-in-companies-like-chatgpt---claude-6b/"))
print("-" * 50)
print("🔍 Testing save_report...")
print(save_report("Great Nicobar Project", "The Great Nicobar Project seeks to transform Great Nicobar into a strategic maritime and economic hub by leveraging its proximity (about 40 nautical miles) to the East–West shipping route and reducing dependence on foreign transshipment ports keeping in view the defense and National Security purpose.\
It includes major infrastructure components: a 14.2 million twenty foot equivalent unit( MTEU) International Container Transshipment Terminal, a Greenfield International Airport (4000 Peak Hour Passengers-PHP)., a 450 MVA gas–solar power plant, and a planned township.\
The development follows a regulated environmental framework, with clearance under the EIA Notification, 2006 and ICRZ Notification, 2019, 42 compliance conditions, diversion of 1.82% of island forest cover, and compensatory afforestation planned over 97.30 sq. km.\
Tribal welfare remains central, with no displacement proposed for Shompen and Nicobarese communities and a net increase in notified tribal reserve area through re-notification measures."))
print("✅ All three tools ready!")


🔍 Testing search_web...
1. **2026 Iran war | Explained, United States, Israel, Strait of ...**
   *   [Conflict](https://www.britannica.com/event/2026-Iran-war#ref473494). [![Image 3: 2026 Iran war](https://cdn.britannica.com/25/285425-004-F2BCEE31/2026-iran-conflict-map.jpg)](https://cdn.britannica.com/25/285425-050-08791D48/2026-iran-conflict-map.jpg)[![Image 4: Key sites of Iran's military-industrial complex](https://cdn.britannica.com/67/273867-049-FAC55FF5/military-and-nuclear-targets-in-
   URL: https://www.britannica.com/event/2026-Iran-war

2. **2026 Iran war - Wikipedia**
   [Jump to content](https://en.wikipedia.org/wiki/2026_Iran_war#bodyContent). *   [(Top)](https://en.wikipedia.org/wiki/2026_Iran_war#). *   [2 Prelude](https://en.wikipedia.org/wiki/2026_Iran_war#Prelude). *   [3.1 First week (28 February – 6 March)](https://en.wikipedia.org/wiki/2026_Iran_war#First_week_(28_February_%E2%80%93_6_March)). *   [3.2 Second week (7–13 March)](https://en.wikipedia.org/w
   URL: 

In [ ]:
# ══════════════════════════════════════════════════════════
# TOOL DEFINITIONS — The "API specs" the LLM reads
# ══════════════════════════════════════════════════════════

agent_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the internet for current information on any topic. Returns titles, snippets and URLs. Use this as your first step when researching.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query — be specific for better results"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Number of results (default 5, max 10)",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_webpage_text",
            "description": "Fetch the full text content of a specific webpage URL. Use this when a search result looks promising and you need more detail.",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {
                        "type": "string",
                        "description": "The full URL of the webpage to read"
                    }
                },
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_report",
            "description": "Save the final research report to a markdown file. Use this ONLY when you have completed all research and are ready to deliver the final output.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "Title of the report"
                    },
                    "content": {
                        "type": "string",
                        "description": "Full report content in markdown format"
                    }
                },
                "required": ["title", "content"]
            }
        }
    }
]

# Map function names to actual functions
tool_functions = {
    "search_web": search_web,
    "get_webpage_text": get_webpage_text,
    "save_report": save_report,
}

print(f"✅ {len(agent_tools)} tools defined and mapped!")
for t in agent_tools:
    print(f"   🔧 {t['function']['name']}: {t['function']['description'][:60]}...")


✅ 3 tools defined and mapped!
   🔧 search_web: Search the internet for current information on any topic. Re...
   🔧 get_webpage_text: Fetch the full text content of a specific webpage URL. Use t...
   🔧 save_report: Save the final research report to a markdown file. Use this ...


In [ ]:
# ══════════════════════════════════════════════════════════
# THE AGENT LOOP — This is the core of EVERY AI agent
# ══════════════════════════════════════════════════════════

def run_agent(user_query: str, max_iterations: int = 10, verbose: bool = True):
    """
    Run the research agent with an autonomous loop.

    The agent will:
    1. Read the user's query
    2. Decide what tool to call (or respond directly)
    3. Execute the tool
    4. Feed results back to itself
    5. Repeat until done or max_iterations reached
    """

    # ── SYSTEM PROMPT: The agent's "personality" and instructions ──

    system_prompt = f"""
You are a bounded research agent for a live educational workshop.

Your job is to answer the user's research question using web evidence.
You must use tools before making claims.

RESEARCH POLICY:
1. Start with a web search.
2. Prefer specific pages over generic search result pages.
3. Read at least 2 relevant sources before writing the report.
4. Do not invent salaries, companies, job counts, or trends.
5. If the evidence is weak, say the evidence is weak.
6. Separate facts from recommendations.
7. Every major claim must include a source URL.
8. Do not include a top-level # title inside the report content. The save_report tool adds the title.

REPORT FORMAT:
## Executive Summary
3-5 bullets with the most important findings.

## Sources Checked
A table with Source, What it was used for, and URL.

## Key Findings
For each finding:
- Claim
- Evidence
- Source URL
- Confidence: High / Medium / Low

## City-wise / Category-wise Breakdown
Use a table if relevant.

## Recommendations for a Student
Give a practical 30-60-90 day action plan.

## Limitations
Mention what could not be verified or where sources were weak.

Current date: {datetime.now().strftime('%Y-%m-%d')}
"""

    # ── CONVERSATION MEMORY: This carries the full history ──
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]

    if verbose:
        print("=" * 60)
        print(f"🚀 AGENT STARTED")
        print(f"📋 Goal: {user_query}")
        print("=" * 60)

    # ── THE LOOP ──
    for iteration in range(1, max_iterations + 1):
        if verbose:
            print(f"\n{'─' * 60}")
            print(f"🔄 Iteration {iteration}/{max_iterations}")
            print(f"{'─' * 60}")

        # Call the LLM with full conversation history + tools
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=agent_tools,
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message

        # ── DECISION POINT: Did the LLM call a tool or give a final answer? ──

        if assistant_message.tool_calls:
            # The LLM wants to use tools — process each one
            messages.append(assistant_message)

            for tool_call in assistant_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                if verbose:
                    args_preview = json.dumps(func_args)
                    if len(args_preview) > 100:
                        args_preview = args_preview[:100] + "..."
                    print(f"   🔧 Calling: {func_name}({args_preview})")

                # Execute the tool
                try:
                    result = tool_functions[func_name](**func_args)
                except Exception as e:
                    result = f"Error executing {func_name}: {str(e)}"

                if verbose:
                    preview = result[:150].replace("\n", " ")
                    print(f"   📄 Result: {preview}...")

                # Add tool result to conversation memory
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        else:
            # No tool calls — the LLM has its final answer
            final_answer = assistant_message.content

            if verbose:
                print(f"\n{'=' * 60}")
                print(f"✅ AGENT FINISHED after {iteration} iterations")
                print(f"{'=' * 60}")

            return {
                "answer": final_answer,
                "iterations": iteration,
                "messages": messages  # Full conversation log
            }

    # If we hit max iterations
    if verbose:
        print(f"\n⚠️ Max iterations ({max_iterations}) reached!")
    return {
        "answer": "Agent reached maximum iterations without completing.",
        "iterations": max_iterations,
        "messages": messages
    }

print("✅ Agent loop defined! Ready to run.")


✅ Agent loop defined! Ready to run.


In [ ]:
# ══════════════════════════════════════════════════════════
# 🚀 RUN THE AGENT — Watch it think and act autonomously!
# ══════════════════════════════════════════════════════════

result = run_agent(
    """Research the current state of AI job market in India in 2026.
    I want to know:
    1. What are the most in-demand AI roles?
    2. What salary ranges are companies offering?
    3. What skills should a fresh graduate focus on?

    Search thoroughly and save a comprehensive report.""",
    max_iterations=10,
    verbose=True
)

print("\n" + "=" * 60)
print("📝 FINAL RESPONSE:")
print("=" * 60)
print(result["answer"])


🚀 AGENT STARTED
📋 Goal: Research the current state of AI job market in India in 2026.
    I want to know:
    1. What are the most in-demand AI roles?
    2. What salary ranges are companies offering?
    3. What skills should a fresh graduate focus on?

    Search thoroughly and save a comprehensive report.

────────────────────────────────────────────────────────────
🔄 Iteration 1/10
────────────────────────────────────────────────────────────
   🔧 Calling: search_web({"query": "AI job market India 2026 in-demand roles salary skills", "max_results": 10})
   📄 Result: 1. **India's AI Job Market to Explode by 2026: Top 10 In-Demand Roles | Abhishek R. Sharma posted on the topic | LinkedIn**    1️⃣ AI Engineer 🔹 Build...
   🔧 Calling: search_web({"query": "current salary AI roles India 2026", "max_results": 10})
   📄 Result: 1. **AI Engineer Salary in India 2026: Trends, Insights & Earnings Guide**    Emerging AI Role Salaries in 2026 ; GenAI Engineer, $174,000, ₹28L – ₹75...
   🔧 Calli

In [ ]:
# Inspect the agent's journey
tool_calls_made = []
for msg in result["messages"]:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            tool_calls_made.append({
                "tool": tc.function.name,
                "args": args
            })

print(f"📊 Agent Statistics:")
print(f"   Total iterations: {result['iterations']}")
print(f"   Tool calls made: {len(tool_calls_made)}")
print(f"   Messages in memory: {len(result['messages'])}")
print()
print("📞 Tool Call Sequence:")
for i, tc in enumerate(tool_calls_made, 1):
    args_str = json.dumps(tc['args'])
    if len(args_str) > 80:
        args_str = args_str[:80] + "..."
    print(f"   {i}. {tc['tool']}({args_str})")


📊 Agent Statistics:
   Total iterations: 7
   Tool calls made: 12
   Messages in memory: 20

📞 Tool Call Sequence:
   1. search_web({"query": "AI job market India 2026", "max_results": 10})
   2. get_webpage_text({"url": "https://medium.com/@ashermachado627/how-ai-is-redefining-indias-job-mar...)
   3. get_webpage_text({"url": "https://www.linkedin.com/pulse/real-impact-ai-indias-workforce-2026-bey...)
   4. get_webpage_text({"url": "https://www.forbes.com/councils/forbestechcouncil/2026/02/19/how-ai-and...)
   5. get_webpage_text({"url": "https://www.indianexpress.com/article/technology/artificial-intelligenc...)
   6. get_webpage_text({"url": "https://www.dqindia.com/news/ai-job-market-impact-2026-ai-may-not-kill-...)
   7. get_webpage_text({"url": "https://www.forbes.com/councils/forbestechcouncil/2026/02/19/how-ai-and...)
   8. search_web({"query": "AI jobs salaries in India 2026", "max_results": 10})
   9. get_webpage_text({"url": "https://www.analyticsvidhya.com/blog/2023/05/ai-e

In [ ]:
# ══════════════════════════════════════════════════════════
# 🔬 DEEP INSPECTION: What did the agent actually do?
# ══════════════════════════════════════════════════════════

# ─── PART 1: HIGH-LEVEL STATISTICS ─────────────────────────
tool_calls_made = []
for msg in result["messages"]:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            tool_calls_made.append({
                "tool": tc.function.name,
                "args": args
            })

print("=" * 70)
print("📊 AGENT STATISTICS")
print("=" * 70)
print(f"   Total iterations: {result['iterations']}")
print(f"   Tool calls made:  {len(tool_calls_made)}")
print(f"   Messages in memory: {len(result['messages'])}")
print()


# ─── PART 2: TOOL CALL SEQUENCE (just the actions) ─────────
print("=" * 70)
print("📞 TOOL CALL SEQUENCE — What the agent decided to DO")
print("=" * 70)
for i, tc in enumerate(tool_calls_made, 1):
    args_str = json.dumps(tc['args'])
    if len(args_str) > 1000:
        args_str = args_str[:1000] + "..."
    print(f"   {i}. {tc['tool']}({args_str})")
print()


# ─── PART 3: ITERATION-BY-ITERATION BREAKDOWN ──────────────
# Group messages by iteration. An iteration = one LLM call + its tool results.
# We detect a new iteration when we see an assistant message.

print("=" * 70)
print("🔄 ITERATION BREAKDOWN — What happened at each step")
print("=" * 70)

iteration = 0
for msg in result["messages"]:
    # Get the role — works for both dict messages and OpenAI message objects
    role = msg["role"] if isinstance(msg, dict) else msg.role

    if role == "system":
        continue  # Skip system prompt for clarity

    if role == "user":
        content = msg["content"] if isinstance(msg, dict) else msg.content
        print(f"\n👤 USER (initial query):")
        preview = content[:200] + "..." if len(content) > 200 else content
        print(f"   {preview}")

    elif role == "assistant":
        iteration += 1
        print(f"\n┌─── 🔄 Iteration {iteration} ───────────────────────────────────")

        # Did the assistant call tools or give a final answer?
        tool_calls = msg.tool_calls if hasattr(msg, 'tool_calls') else msg.get('tool_calls')

        if tool_calls:
            print(f"│ 🤖 ASSISTANT decided to call {len(tool_calls)} tool(s):")
            for tc in tool_calls:
                args = json.loads(tc.function.arguments)
                args_preview = json.dumps(args)
                if len(args_preview) > 100:
                    args_preview = args_preview[:100] + "..."
                print(f"│    🔧 {tc.function.name}({args_preview})")
        else:
            content = msg.content if hasattr(msg, 'content') else msg.get('content', '')
            print(f"│ 🤖 ASSISTANT gave FINAL ANSWER (no more tool calls):")
            preview = content[:300].replace("\n", " ")
            if len(content) > 300:
                preview += "..."
            print(f"│    {preview}")

    elif role == "tool":
        content = msg["content"] if isinstance(msg, dict) else msg.content
        preview = content[:150].replace("\n", " ")
        if len(content) > 150:
            preview += "..."
        print(f"│ 📄 TOOL RESULT: {preview}")
        print("└" + "─" * 60)

print()


# ─── PART 4: FULL MEMORY DUMP (every message in conversation) ──
print("=" * 70)
print("🧠 FULL MEMORY DUMP — Every message the LLM saw")
print("=" * 70)
print("(This is the agent's complete 'memory' — what it knows about the conversation)")
print()

for i, msg in enumerate(result["messages"]):
    role = msg["role"] if isinstance(msg, dict) else msg.role

    print(f"─── Message #{i} | role: {role.upper()} ───")

    if role == "system":
        content = msg["content"] if isinstance(msg, dict) else msg.content
        print(f"   System Prompt: {content[:200]}...")

    elif role == "user":
        content = msg["content"] if isinstance(msg, dict) else msg.content
        print(f"   {content[:300]}{'...' if len(content) > 300 else ''}")

    elif role == "assistant":
        tool_calls = msg.tool_calls if hasattr(msg, 'tool_calls') else msg.get('tool_calls')
        content = msg.content if hasattr(msg, 'content') else msg.get('content')

        if tool_calls:
            print(f"   [Decided to call {len(tool_calls)} tool(s)]")
            for tc in tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"      → {tc.function.name}({json.dumps(args)[:120]})")
        if content:
            print(f"   Text: {content[:300]}{'...' if len(content) > 300 else ''}")

    elif role == "tool":
        content = msg["content"] if isinstance(msg, dict) else msg.content
        tcid = msg["tool_call_id"] if isinstance(msg, dict) else msg.tool_call_id
        print(f"   [Result for tool_call_id: {tcid}]")
        print(f"   {content[:250]}{'...' if len(content) > 250 else ''}")

    print()

print("=" * 70)
print(f"✅ Inspection complete. The agent saw {len(result['messages'])} messages total.")
print("=" * 70)

📊 AGENT STATISTICS
   Total iterations: 7
   Tool calls made:  12
   Messages in memory: 20

📞 TOOL CALL SEQUENCE — What the agent decided to DO
   1. search_web({"query": "AI job market India 2026", "max_results": 10})
   2. get_webpage_text({"url": "https://medium.com/@ashermachado627/how-ai-is-redefining-indias-job-market-in-2026-de10c63f157a"})
   3. get_webpage_text({"url": "https://www.linkedin.com/pulse/real-impact-ai-indias-workforce-2026-beyond-hype-meraquii-cw3nc"})
   4. get_webpage_text({"url": "https://www.forbes.com/councils/forbestechcouncil/2026/02/19/how-ai-and-gccs-will-impact-indias-job-market-in-2026/"})
   5. get_webpage_text({"url": "https://www.indianexpress.com/article/technology/artificial-intelligence/india-sees-fastest-global-growth-in-ai-hiring-at-59-5-linkedin-report-10655284/"})
   6. get_webpage_text({"url": "https://www.dqindia.com/news/ai-job-market-impact-2026-ai-may-not-kill-jobs-but-it-could-slow-hiring-says-new-report-11461663"})
   7. get_webpage_t

In [ ]:
# 🎯 TRY DIFFERENT QUERIES — Uncomment one and run!

# Example 1: Person research
# result = run_agent("Research Satya Nadella's key decisions and impact as Microsoft CEO in 2024-2026. Save a report.")

# Example 2: Tech comparison
# result = run_agent("Compare React vs Next.js vs Svelte for building web apps in 2026. Which should a new developer learn? Save a report.")

# Example 3: Career advice
result = run_agent(
    "Research the top 10 companies hiring for AI/ML roles in India right now. "
    "Include company names, roles, and any salary info you can find. Save a report.",
    verbose=True
)

print("\n📝 Final Answer:")
print(result["answer"][:500] + "..." if len(result["answer"]) > 500 else result["answer"])


🚀 AGENT STARTED
📋 Goal: Research the top 10 companies hiring for AI/ML roles in India right now. Include company names, roles, and any salary info you can find. Save a report.

────────────────────────────────────────────────────────────
🔄 Iteration 1/10
────────────────────────────────────────────────────────────
   🔧 Calling: search_web({"query": "top companies hiring AI ML jobs in India May 2026", "max_results": 10})
   📄 Result: Error executing search_web: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))...

────────────────────────────────────────────────────────────
🔄 Iteration 2/10
────────────────────────────────────────────────────────────
   🔧 Calling: search_web({"query": "top companies hiring AI ML roles in India May 2026", "max_results": 10})
   📄 Result: 1. **Best AI Careers in India 2026: High-Paying Jobs, Skills & Roadmap**    By clicking the button, I accept the Terms of Use of the service and its P...

──────────────────────

In [ ]:
# ══════════════════════════════════════════════════════════
# ✅ SOLUTION (Don't peek until you've tried!)
# ══════════════════════════════════════════════════════════

def search_jobs(role: str, location: str = "India") -> str:
    """Search for job listings matching the given role and location."""
    query = f"{role} jobs {location} hiring 2026"
    results = search_web(query, max_results=5)
    return results

search_jobs_tool = {
    "type": "function",
    "function": {
        "name": "search_jobs",
        "description": "Search for current job listings and opportunities for a specific role and location. Use when the user asks about job openings, hiring, or career opportunities.",
        "parameters": {
            "type": "object",
            "properties": {
                "role": {
                    "type": "string",
                    "description": "The job title or role to search for (e.g., 'ML Engineer', 'Data Scientist')"
                },
                "location": {
                    "type": "string",
                    "description": "Preferred job location",
                    "default": "India"
                }
            },
            "required": ["role"]
        }
    }
}

# Add to our tools list and function map
enhanced_tools = agent_tools + [search_jobs_tool]
enhanced_functions = {**tool_functions, "search_jobs": search_jobs}

print("✅ New tool added! The agent now has these capabilities:")
for t in enhanced_tools:
    print(f"   🔧 {t['function']['name']}")


✅ New tool added! The agent now has these capabilities:
   🔧 search_web
   🔧 get_webpage_text
   🔧 save_report
   🔧 search_jobs


In [ ]:
# ══════════════════════════════════════════════════════════
# 🚀 RUN ENHANCED AGENT — Now with job search!
# ══════════════════════════════════════════════════════════

def run_enhanced_agent(user_query: str, max_iterations: int = 10, verbose: bool = True):
    """Same agent loop but with our enhanced tool set."""

    system_prompt = f"""You are a career research agent. You help users find jobs
and research career opportunities. You have tools to search the web,
read web pages, search for specific jobs, and save reports.

STRATEGY:
1. Search for relevant jobs using the search_jobs tool
2. Optionally search the web for more context (salary data, company info)
3. Read promising pages for details
4. Save a comprehensive career report

Current date: {datetime.now().strftime('%Y-%m-%d')}"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]

    if verbose:
        print(f"🚀 ENHANCED AGENT STARTED | Goal: {user_query[:80]}...")
        print("=" * 60)

    for iteration in range(1, max_iterations + 1):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=enhanced_tools,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                func_name = tc.function.name
                func_args = json.loads(tc.function.arguments)
                if verbose:
                    print(f"   🔧 [{iteration}] {func_name}({json.dumps(func_args)[:80]}...)")

                try:
                    result = enhanced_functions[func_name](**func_args)
                except Exception as e:
                    result = f"Error: {str(e)}"

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })
        else:
            if verbose:
                print(f"\n✅ Done in {iteration} iterations!")
            return {"answer": msg.content, "iterations": iteration}

    return {"answer": "Max iterations reached.", "iterations": max_iterations}


# Run it!
result = run_enhanced_agent(
    "I'm a fresh CS graduate interested in AI/ML. "
    "Find me relevant job openings in Bangalore and Delhi. "
    "Also research what skills I should highlight. Save a complete report.",
    verbose=True
)

print("\n" + "=" * 60)
print("📝 REPORT:")
print("=" * 60)
print(result["answer"])


🚀 ENHANCED AGENT STARTED | Goal: I'm a fresh CS graduate interested in AI/ML. Find me relevant job openings in Ba...
   🔧 [1] search_jobs({"role": "AI/ML Engineer", "location": "Bangalore"}...)
   🔧 [1] search_jobs({"role": "AI/ML Engineer", "location": "Delhi"}...)
   🔧 [2] get_webpage_text({"url": "https://www.glassdoor.co.in/Job/bengaluru-ai-or-ml-engineer-jobs-SRCH_I...)
   🔧 [2] get_webpage_text({"url": "https://www.glassdoor.co.in/Job/jobs.htm?sc.occupationParam=AI%2FML+Eng...)
   🔧 [3] get_webpage_text({"url": "https://internshala.com/jobs/machine-learning-jobs-in-bangalore/"}...)
   🔧 [3] get_webpage_text({"url": "https://internshala.com/jobs/machine-learning-jobs-in-delhi/"}...)
   🔧 [4] search_web({"query": "skills needed for AI ML engineer fresh graduate 2026", "max_results":...)
   🔧 [5] get_webpage_text({"url": "https://wininlifeacademy.com/ai-engineer-skills-required/"}...)
   🔧 [5] get_webpage_text({"url": "https://www.datacamp.com/blog/essential-ai-engineer-skills"}...)